<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will compute a tiny, reproducible metric: <b>exact-center coverage</b> on a toy train/test split.
</div>

# S07 · Micro-evaluation: exact-center coverage


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: define a simple coverage metric
- Hands-on: train/test split + compute coverage


# Theory

We define an intentionally strict metric:
- Extract the (created, deleted) bond pairs as a "center signature"
- A test reaction is "covered" if its center signature appeared in training

This is a teaching metric: easy to compute and interpret.


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
from random import Random

df = pd.read_csv("data/reactions_mapped.csv")
rnd = Random(0)
idx = list(range(len(df)))
rnd.shuffle(idx)

split = max(1, int(0.7 * len(df)))
train = df.iloc[idx[:split]].reset_index(drop=True)
test  = df.iloc[idx[split:]].reset_index(drop=True)

print("Train:", len(train), "Test:", len(test))


In [ ]:
def center_signature(am_rxn_smiles: str):
    react, prod = am_rxn_smiles.split(">>")
    mR = Chem.MolFromSmiles(react)
    mP = Chem.MolFromSmiles(prod)

    def pairs(m):
        out=set()
        for b in m.GetBonds():
            ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
            if ai and aj:
                out.add((min(ai,aj), max(ai,aj)))
        return out

    pR = pairs(mR); pP = pairs(mP)
    created = tuple(sorted(pP - pR))
    deleted = tuple(sorted(pR - pP))
    return created, deleted

train_sigs = [center_signature(r.am_rxn_smiles) for _, r in train.iterrows()]

covered = 0
for _, r in test.iterrows():
    if center_signature(r.am_rxn_smiles) in train_sigs:
        covered += 1

metrics = {
    "train_size": len(train),
    "test_size": len(test),
    "covered": covered,
    "coverage_exact_center": covered / max(1, len(test)),
}
metrics


# Discussion
- Exact-center coverage underestimates practical generalization (too strict).
- Still, it's useful to check that you extract consistent centers and rules.


# Quiz
1. Suggest a softer metric than exact-center match.
2. Why might exact-center match fail even for similar reactions?
3. How would you incorporate bond order changes into the signature?


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
